## 제출 정보

- 이름: 예민호

- GitHub ID: yegamanse-cpu

- 작성일: 2026.11.10

- 최종 제출 URL: https://github.com/yegamanse-cpu/llm-data-analysis-study/blob/main/chapter09/chapter09.ipynb

In [6]:
from pathlib import Path
import sys
import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트 폴더를 찾을 수 없습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

PROJECT_ROOT: C:\dev\llm-data-analysis-course
PROCESSED_DIR: C:\dev\llm-data-analysis-course\data\processed


- ch07 때처럼 src 모듈 불러오기 오류 발생. 노트북이 notebooks/ch09에 있어 위로 올라가면서 src 폴더가 있는 곳을 찾도록 수정해서 해결
- 실행 전에 python scripts/prepare_ch09_data.py 실행하여 processed 파일 만들어 둠

## 1. 예측 문제 정의

- Target: order_total

- 예측 단위 : 주문 1건

- 예측 시점 : 고객이 상품을 고르기 전

- 예측 시점에 알 수 있는 정보: 오늘 날짜, 고객 나이·성별·도시

- 예측 시점에 알 수 없는 정보: 수량, 단가, line_total, 고른 상품, 주문 상태, 결제수단

### 나의 해석과 판단

- 주문을 완료하면 금액이 다 나와 예측할 게 없으므로 상품 고르기 전 시점으로 설정

In [7]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

Target: order_total
Allowed features: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']
Forbidden features: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'product_id', 'quantity', 'total_quantity', 'unit_price']
Feature leakage gate: PASS


In [9]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

customers (150, 6)
orders (300, 7)
order_items (764, 6)


In [10]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
display(model_data.head())
print('Model rows:', len(model_data))

,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek,order_total,gender,age,city
0,238,124,2025-09-15,bank_transfer,completed,9,0,361000,M,67,대구
1,79,62,2025-09-17,bank_transfer,refunded,9,2,283000,F,63,광주
2,12,98,2025-09-18,kakao_pay,completed,9,3,444000,F,61,인천
3,169,9,2025-09-18,kakao_pay,completed,9,3,448000,M,69,서울
4,152,62,2025-09-19,naver_pay,refunded,9,4,985000,F,63,광주


Model rows: 300


In [8]:
feature_audit = build_feature_audit()
display(feature_audit)

,column,selected,role,reason
0,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,avg_unit_price,False,forbidden,주문 상세에서 만든 목표 대리 변수
7,customer_id,False,forbidden,고객 식별자
8,item_count,False,forbidden,주문 상세가 확인된 뒤 계산되는 사후 집계
9,line_total,False,forbidden,목표값을 구성하는 주문 상세 금액


## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| order_month | O | X | X | O | 오늘 날짜 |
| order_dayofweek | O | X | X | O | 오늘 날짜 |
| age | O | X | X | O | 가입 정보 |
| gender | O | X | X | O | 가입 정보 |
| city | O | X | X | O | 가입 정보 |
| payment_method | O | X | O | O | 사용가정 |
| quantity | X | O | X | X | 금액 계산용 |
| unit_price | X | O | X | X | 금액 계산용 |
| line_total | X | O | X | X | target |
| item_count / total_quantity / avg_unit_price | X | O | O | X | 대리변수, 사후집계 |
| order_status | X | X | O | X | 사후집계 |
| order_id / customer_id / product_id | - | X | O | X | 큰 의미 없음 |

- 금지 feature overlap 개수: 0

- 가장 위험하다고 판단한 feature: total_quantity

- 이유: 주문 상세가 다 확정돼야 계산되는 값으로 쓸 수 없음

![Leakage 점검](images/01.png)

In [ ]:
# line_total = quantity × unit_price 확인
items = data['order_items']
mismatch = (items['line_total'] != items['quantity'] * items['unit_price']).sum()
print('order_items 행 수:', len(items))
print('line_total 불일치:', mismatch)

# 주문 300건이 전부 상세를 갖고 있는지
print('orders 행 수:', len(data['orders']))
print('상세 있는 주문 수:', data['orders']['order_id'].isin(items['order_id']).sum())
print('모델 데이터 행 수:', len(model_data))

order_items 행 수: 764
line_total 불일치: 0
orders 행 수: 300
상세 있는 주문 수: 300
모델 데이터 행 수: 300


- line_total 및 orders ↔ target, orders → customers 검증을 위해 LLM으로 코딩하여 데이터 확인

## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity × unit_price
order_total = 같은 order_id의 line_total 합계
```

- line_total 검증 결과: 불일치 0 

- orders ↔ target 관계 검증 결과: one_to_one PASS. 주문 300건 전부 상세 있음

- orders → customers 관계 검증 결과: many_to_one PASS. 못 찾는 주문 0

### 나의 해석과 판단

- 가이드 노트북에는 관계를 직접 눈으로 확인하는 코드가 없어 판단이 되지 않았음. LLM으로 코드를 만들어 직접 눈으로 확인이 필요했음

In [14]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

,split,rows,ratio_pct,start_date,end_date
0,train,244,81.33,2025-09-15,2026-06-20
1,test,56,18.67,2026-06-21,2026-09-14


Chronological split gate: PASS


In [19]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

models = make_regression_models(random_state=42)
print(list(models))

Train: (244, 6) Final Test: (56, 6)
['Baseline Mean', 'Linear Regression', 'Random Forest']


## 4. Train / Final Test 분할

- Train 시작일: 2025-09-15

- Train 종료일: 2026-06-20

- Train 행 수: 244 (81.3%)

- Final Test 시작일: 2026-06-21

- Final Test 종료일: 2026-09-14

- Final Test 행 수: 56 (18.7%)

- 같은 달력 날짜 중복 여부: 없음 

### 분할 판단

- 지난 주문으로 학습하여 앞으로 들어올 주문을 맞추는 구조로 시간 순서로 분할이 적정하다고 판단

![시간 분할](images/02.png)

In [20]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,423004.808743,44153.043466,-0.022821
1,Random Forest,5,434988.775913,23875.590913,-0.110935
2,Linear Regression,5,476605.270739,74179.085725,-0.392969


## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | 423,005 | 44,153 | -0.023 | 평균값 |
| Random Forest | 434,989 | 23,876 | -0.111 | MAE 제일 낮음 |
| Linear Regression | 476,605 | 74,179 | -0.393 | - |


### Train CV로 선택한 비베이스라인 모델

- Selected Model: Random Forest

- 선택 기준: Linear Regression보다 CV MAE 평균 낮음

- Final Test 결과를 보기 전에 선택했는가: 예

### 나의 해석과 판단

- Random Forest이 표준편차가 제일 작으나 평균은 Baseline이 약 12,000원 낮음

In [21]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

Frozen selected model: Random Forest


In [22]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.699620,-0.001121,0.00
1,Random Forest,selected_by_train_cv,327645.945199,464694.076952,550431.830272,-0.112339,-4.79


## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 416,266 | 443,450 | 522,190 | -0.001 | 0.00% |
| Random Forest | selected_by_train_cv | 327,646 | 464,694 | 550,432 | -0.112 | -4.79% |

![최종 회귀 평가](images/03.png)

### 결과 관찰

- Random Forest가 Baseline보다 test MAE가 약 21,000원 크며 개선율 -4.79%로 낮아짐

### 나의 해석과 판단

- Frozen Model이 baseline보다 실제로 개선되었는가? → 개선되지 않음

- MAE와 RMSE 차이에서 무엇을 볼 수 있는가? → RMSE가 MAE보다 약 80,000원 정도 크게 나타남

- R²는 어떻게 해석할 수 있는가? → 둘 다 0 근처로 현 feature로는 주문 금액 차이를 설명 못 함

### 업무·분석적 의미

- 고객 정보·날짜만으론 주문을 예측하기 어렵다고 판단됨

In [24]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')

,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-08-22,1877000,5.415905e+05,1.335410e+06,1.335410e+06,Random Forest
1,21,2026-08-26,1767000,7.336590e+05,1.033341e+06,1.033341e+06,Random Forest
2,145,2026-08-03,1863000,8.557899e+05,1.007210e+06,1.007210e+06,Random Forest
3,96,2026-07-18,1690000,6.979246e+05,9.920754e+05,9.920754e+05,Random Forest
4,183,2026-07-30,1685000,7.835840e+05,9.014160e+05,9.014160e+05,Random Forest
5,15,2026-07-30,1677000,8.191082e+05,8.578918e+05,8.578918e+05,Random Forest
6,168,2026-08-18,10000,8.527183e+05,-8.427183e+05,8.427183e+05,Random Forest
7,282,2026-07-01,1566000,7.388317e+05,8.271683e+05,8.271683e+05,Random Forest
8,194,2026-07-26,257000,1.082731e+06,-8.257306e+05,8.257306e+05,Random Forest
9,179,2026-09-14,86000,8.390004e+05,-7.530004e+05,7.530004e+05,Random Forest


주의: order_id가 포함된 이 표는 내부 진단용입니다.


## 7. 대표 오차 사례

### 사례 1

- 실제값: 1,877,000

- 예측값: 541,591

- residual: 1,335,410

- absolute error: 1,335,410

- 관찰: 실 금액과 예측값이 약 3배 이상 차이남(과소예측)

- 가능한 이유 후보: 이와 같은 훈련사례가 없는것이 원인일 수 있을 거라고 판단됨

- 추가 확인: 이전 주문 금액이 평소에도 컸는지

### 사례 2

- 실제값: 10,000

- 예측값: 852,718

- residual: -842,718

- absolute error: 842,718

- 관찰: 실 금액은 제일 작은 수준이나 평균으로 예측

- 가능한 이유 후보: 상품 수나 가격을 모르기 떄문에 이런 경우는 예측이 어렵다고 판단

- 추가 확인: 비슷한 케이스의 주문이 얼마나 자주 나오는지

### 사례 3

- 실제값: 257,000

- 예측값: 1,082,731

- residual: -825,731

- absolute error: 825,731

- 관찰: 실 금액보다 예측값이 약 4배 큼

- 가능한 이유 후보: 비슷한 유형의 패턴을 따라 예측한 것일 수 있음

- 추가 확인: train에서 비슷한 조합 주문 몇 건이었는지

원인을 데이터가 직접 보여 주지 않는다면 사실처럼 단정하지 않습니다.

In [25]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

### FAIL 항목

- 없음

- 해당없음

In [26]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

model_data_internal -> C:\dev\llm-data-analysis-course\reports\ch09_regression_model_data_internal.csv
split_summary -> C:\dev\llm-data-analysis-course\reports\ch09_regression_split_summary.csv
feature_audit -> C:\dev\llm-data-analysis-course\reports\ch09_regression_feature_audit.csv
cv_summary -> C:\dev\llm-data-analysis-course\reports\ch09_regression_cv_summary.csv
model_comparison -> C:\dev\llm-data-analysis-course\reports\ch09_regression_model_comparison.csv
predictions_internal -> C:\dev\llm-data-analysis-course\reports\ch09_regression_predictions_internal.csv
validation -> C:\dev\llm-data-analysis-course\reports\ch09_regression_validation.csv
checklist -> C:\dev\llm-data-analysis-course\reports\ch09_regression_checklist.csv
report -> C:\dev\llm-data-analysis-course\reports\ch09_regression_report.md
actual_vs_predicted_figure -> C:\dev\llm-data-analysis-course\reports\figures\ch09_actual_vs_predicted.png
residual_figure -> C:\dev\llm-data-analysis-course\reports\figures\ch09_resid

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
- order_id가 있으면 주문데이터를 재추출 가능하므로 제외(Internal로 구분)

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거

1. MAE가 baseline보다 4.79% 낮음

2. Train CV에서도 baseline이 Random Forest보다 좋음

3. R²가 -0.11로 큰 차이가 없는 것으로 판단

### 업무적 위험

- 이 모델로 예상 시 평균 46만원씩 틀린 값을 사용하게 되므로 사용 보류

### 현재 데이터의 한계

- 전체 300건, test 56건으로 데이터가 적고 train 기간에 7·8월이 없어 월 패턴을 배울 수 없음

### 다음 개선 우선순위

1. 과거 주문 평균 금액 같은 이력 feature 추가 

2. completed 주문만으로 다시 해보고 결과 비교

3. 데이터 1년치 이상 모은 뒤 같은 절차로 재평가

In [27]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

Selected by Train CV: Random Forest


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/prepare_ch09_data.py
python scripts/run_regression_analysis.py
```

- 실행 성공 여부: O
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: PASS
- Notebook 결과와 스크립트 결과의 일관성: CV, test MAE 모두동일

## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다.